# 02 - Ingest Documents into Chroma

This notebook parses PDFs, PubMed XML, and FDA JSON, performs section-aware chunking, generates embeddings, and upserts into a Chroma vectorstore.

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
from pathlib import Path
import logging
import os
logging.basicConfig(level=logging.INFO)

from biotech_rag.data.pdf_processor import pdf_to_sections
from biotech_rag.data.pubmed_parser import parse_pubmed_xml
from biotech_rag.data.fda_parser import parse_fda_json
from biotech_rag.indexing.chunkers import HybridScientificChunker
from biotech_rag.indexing.embedders import Embedder
from biotech_rag.indexing.vectorstore import init_chroma, upsert_documents
from biotech_rag.config import settings


def find_project_root(start: Path) -> Path:
    """Find project root by locating pyproject.toml or .git."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_RAW = PROJECT_ROOT / "data" / "raw"
PDF_DIR = DATA_RAW / "pdf_studies"
XML_DIR = DATA_RAW / "pubmed_semantic_data" / "xml_files"
FDA_DIR = DATA_RAW / "fda_labels"
VSTORE_DIR = PROJECT_ROOT / "data" / "processed" / "vectorstore" / "chroma_db"

# Initialize components (install required packages locally)
chunker = HybridScientificChunker(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    paragraph_separator="\n\n",
    secondary_chunking_regex=r"\n(?=[A-Z][a-z]+:|\d+\.\d+|\*\*)",
    fallback_min_tokens=settings.chunk_size,
    fallback_overlap_tokens=settings.chunk_overlap,
)

# Centralized embedder defaults (OpenRouter qwen first, local fallback if key missing)
embedder = Embedder()
logging.info(f"Embedder backend: {embedder.backend} | model: {embedder.model}")

client, collection = init_chroma(VSTORE_DIR, collection_name="clinical_trials")

logging.info("Components initialized")

INFO:root:Using LlamaIndex SentenceSplitter for semantic chunking.
INFO:root:Embedder backend: openrouter | model: qwen/qwen3-embedding-8b:floor
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:root:Components initialized


In [3]:
# Test OpenRouter qwen embedding (qwen/qwen3-embedding-0.6b)
from dotenv import load_dotenv
load_dotenv()
from biotech_rag.indexing.embedders import Embedder
import os
print('OPENROUTER_BASE_URL=', os.getenv('OPENROUTER_BASE_URL'))
print('OPENROUTER_API_KEY present:', bool(os.getenv('OPENROUTER_API_KEY')))
embedder = Embedder(backend='openrouter', model='qwen/qwen3-embedding-8b:floor', batch_size=8)
texts = ['Test embedding for OpenRouter qwen model', 'Second short test']
embs = embedder.embed(texts)
print('Embeddings returned:', len(embs), 'vectors')
if embs:
    print('Vector dim (len first vector):', len(embs[0]))
    print('Sample first vector (first 6 values):', embs[0][:6])
else:
    print('No embeddings returned — check network/key; local fallback may have been used.')

OPENROUTER_BASE_URL= https://openrouter.ai/api/v1
OPENROUTER_API_KEY present: True
Embeddings returned: 2 vectors
Vector dim (len first vector): 4096
Sample first vector (first 6 values): [-0.0047150528989732265, -0.012751400470733643, 0.014767901040613651, -0.02906133234500885, 0.01334448903799057, -0.036297012120485306]


In [2]:
# Debug: check files, extracted sections, chunking behavior, and torch GPU
from pathlib import Path
import logging
logging.getLogger().setLevel(logging.INFO)

pdf_files = sorted(PDF_DIR.glob('*.pdf')) if PDF_DIR.exists() else []
print('PDF files found:', len(pdf_files))

if pdf_files:
    p = pdf_files[0]
    print('First PDF:', p.name)
    sections = pdf_to_sections(p)
    print('Sections extracted from first PDF:', len(sections))
    
    if sections:
        s0 = sections[0]
        txt = (s0.get('text') or '').strip()
        print('First section text length (chars):', len(txt))
        
        # quick chunking test using the configured chunker
        meta = dict(s0)
        meta.pop('text', None)
        test_chunks = chunker.chunk_text(txt, section_title=s0.get('section_title'), metadata=meta)
        print('Chunks from first section with configured chunker:', len(test_chunks))
        
        if test_chunks:
            print('Example chunk lengths (first 3):', [len(c['text']) for c in test_chunks[:3]])
else:
    print('No PDF files found. Ensure PDFs are placed in data/raw/pdf_studies')

try:
    import torch
    print('torch version:', torch.__version__)  # Fixed: torch.version -> torch.__version__
    print('cuda available:', torch.cuda.is_available())
    
    if torch.cuda.is_available():
        print('cuda device count:', torch.cuda.device_count())
        try:
            print('device name:', torch.cuda.get_device_name(0))
        except Exception as e:
            print('device name error:', e)
    else:
        print('CUDA not available to PyTorch in this environment (sentence-transformers will use CPU).')
        
except Exception as e:
    print('Failed to import torch')

PDF files found: 29
First PDF: NCT03088813.pdf
Sections extracted from first PDF: 86
First section text length (chars): 595
Chunks from first section with configured chunker: 1
Example chunk lengths (first 3): [595]
Failed to import torch


In [5]:
def ingest_sections(sections, source_tag, chunker=chunker, embedder=embedder, collection=collection):
    """Ingest a list of section dicts into Chroma.
    Each section should contain at least 'text' and optional metadata keys (filename, page, section_title, pmid, pmcid, nct_id, drug_name).
    Returns number of chunks upserted.
    """
    docs = []
    # Use a global index for the whole document to ensure unique IDs
    global_chunk_idx = 0
    
    for s in sections:
        text = (s.get('text') or '').strip()
        if not text:
            continue
            
        meta = dict(s)
        meta.pop('text', None)
        
        # Get chunks for this section
        chunks = chunker.chunk_text(text, section_title=meta.get('section_title'), metadata=meta)
        
        for c in chunks:
            chunk_meta = dict(meta)
            chunk_meta.update(c.get('metadata', {}))
            
            # Ensure identifiers are priority in metadata for clinical filtering
            filename = chunk_meta.get('filename', 'unknown')
            page = chunk_meta.get('page', 0)
            nct_id = chunk_meta.get('nct_id')
            drug_name = chunk_meta.get('drug_name')
            
            # Revised ID format: {tag}-{nct_id|drug_name|filename}-p{page}-idx{global_idx}
            primary_id = nct_id or drug_name or filename
            doc_id = f"{source_tag}-{primary_id}-p{page}-idx{global_chunk_idx}"
            
            docs.append({'id': doc_id, 'text': c['text'], 'metadata': chunk_meta})
            global_chunk_idx += 1
            
    if not docs:
        return 0
        
    texts = [d['text'] for d in docs]
    # Check for duplicate IDs before sending to Chroma to be safe
    ids = [d['id'] for d in docs]
    if len(ids) != len(set(ids)):
        import collections
        dupes = [item for item, count in collections.Counter(ids).items() if count > 1]
        logging.warning(f"Duplicate IDs detected in batch: {dupes[:5]}...")
        # dedup docs by ID, keeping the first one
        seen = set()
        unique_docs = []
        for d in docs:
            if d['id'] not in seen:
                unique_docs.append(d)
                seen.add(d['id'])
        docs = unique_docs
        texts = [d['text'] for d in docs]

    embeddings = embedder.embed(texts)
    upsert_documents(collection, docs, embeddings=embeddings)
    return len(docs)

In [6]:
# Ingest PDFs
pdf_files = sorted(PDF_DIR.glob('*.pdf')) if PDF_DIR.exists() else []
pdf_total = 0
for p in pdf_files:
    logging.info(f'Processing PDF: {p.name}')
    sections = pdf_to_sections(p)
    n = ingest_sections(sections, source_tag='pdf')
    logging.info(f'Ingested {n} chunks from {p.name}')
    pdf_total += n

logging.info(f'Total PDF chunks ingested: {pdf_total}')

INFO:root:Processing PDF: NCT03088813.pdf
INFO:root:Ingested 96 chunks from NCT03088813.pdf
INFO:root:Processing PDF: NCT03425643.pdf
INFO:root:Ingested 57 chunks from NCT03425643.pdf
INFO:root:Processing PDF: NCT03447769.pdf
INFO:root:Ingested 24 chunks from NCT03447769.pdf
INFO:root:Processing PDF: NCT03515837.pdf
INFO:root:Ingested 133 chunks from NCT03515837.pdf
INFO:root:Processing PDF: NCT03519971.pdf
INFO:root:Ingested 129 chunks from NCT03519971.pdf
INFO:root:Processing PDF: NCT03521154.pdf
INFO:root:Ingested 32 chunks from NCT03521154.pdf
INFO:root:Processing PDF: NCT03594747.pdf
INFO:root:Ingested 2 chunks from NCT03594747.pdf
INFO:root:Processing PDF: NCT03663205.pdf
INFO:root:Ingested 45 chunks from NCT03663205.pdf
INFO:root:Processing PDF: NCT03703297.pdf
INFO:root:Ingested 21 chunks from NCT03703297.pdf
INFO:root:Processing PDF: NCT03743051.pdf
INFO:root:Ingested 69 chunks from NCT03743051.pdf
INFO:root:Processing PDF: NCT03811002.pdf
INFO:root:Ingested 8 chunks from NCT0

In [7]:
# Ingest PubMed XML files
xml_files = sorted(XML_DIR.glob('*.xml')) if XML_DIR.exists() else []
xml_total = 0
for x in xml_files:
    logging.info(f'Processing XML: {x.name}')
    sections = parse_pubmed_xml(x)
    n = ingest_sections(sections, source_tag='pubmed')
    logging.info(f'Ingested {n} chunks from {x.name}')
    xml_total += n

logging.info(f'Total XML chunks ingested: {xml_total}')

INFO:root:Processing XML: NCT03088813_38648575.xml
INFO:root:Ingested 4 chunks from NCT03088813_38648575.xml
INFO:root:Processing XML: NCT03296163_33914256.xml
INFO:root:Ingested 6 chunks from NCT03296163_33914256.xml
INFO:root:Processing XML: NCT03351361_39486424.xml
INFO:root:Ingested 5 chunks from NCT03351361_39486424.xml
INFO:root:Processing XML: NCT03409614_36008722.xml
INFO:root:Ingested 1 chunks from NCT03409614_36008722.xml
INFO:root:Processing XML: NCT03425643_37272513.xml
INFO:root:Ingested 4 chunks from NCT03425643_37272513.xml
INFO:root:Processing XML: NCT03447769_37788412.xml
INFO:root:Ingested 4 chunks from NCT03447769_37788412.xml
INFO:root:Processing XML: NCT03515837_39173098.xml
INFO:root:Ingested 4 chunks from NCT03515837_39173098.xml
INFO:root:Processing XML: NCT03521154_38828946.xml
INFO:root:Ingested 4 chunks from NCT03521154_38828946.xml
INFO:root:Processing XML: NCT03594747_33792623.xml
INFO:root:Ingested 8 chunks from NCT03594747_33792623.xml
INFO:root:Processin

In [8]:
# Ingest FDA JSON labels
fda_files = sorted(FDA_DIR.glob('*.json')) if FDA_DIR.exists() else []
fda_total = 0
for f in fda_files:
    logging.info(f'Processing FDA file: {f.name}')
    sections = parse_fda_json(f)
    n = ingest_sections(sections, source_tag='fda')
    logging.info(f'Ingested {n} chunks from {f.name}')
    fda_total += n

logging.info(f'Total FDA chunks ingested: {fda_total}')

INFO:root:Processing FDA file: anamorelin hydrochloride_label.json
INFO:root:Ingested 1 chunks from anamorelin hydrochloride_label.json
INFO:root:Processing FDA file: atezolizumab_label.json
INFO:root:Ingested 167 chunks from atezolizumab_label.json
INFO:root:Processing FDA file: bevacizumab_label.json
INFO:root:Ingested 80 chunks from bevacizumab_label.json
INFO:root:Processing FDA file: bintrafusp alfa_label.json
INFO:root:Ingested 51 chunks from bintrafusp alfa_label.json
INFO:root:Processing FDA file: bupivacaine_label.json
INFO:root:Ingested 29 chunks from bupivacaine_label.json
INFO:root:Processing FDA file: canakinumab_label.json
INFO:root:Ingested 3 chunks from canakinumab_label.json
INFO:root:Processing FDA file: carboplatin_label.json
INFO:root:Ingested 47 chunks from carboplatin_label.json
INFO:root:Processing FDA file: cemiplimab_label.json
INFO:root:Ingested 87 chunks from cemiplimab_label.json
INFO:root:Processing FDA file: cetirizine_label.json
INFO:root:Ingested 3 chunk

In [10]:
client, collection = init_chroma(Path("../data/processed/vectorstore/chroma_db"),
                                 collection_name="clinical_trials")

# Persist if supported (older chroma versions)
if hasattr(client, "persist"):
    client.persist()

print("Collection count:", collection.count())
print(collection.get(limit=5, include=["documents", "metadatas"]))

Collection count: 5532
{'ids': ['pdf-NCT03088813-p1-idx0', 'pdf-NCT03088813-p1-idx1', 'pdf-NCT03088813-p1-idx2', 'pdf-NCT03088813-p1-idx3', 'pdf-NCT03088813-p1-idx4'], 'embeddings': None, 'documents': ['Original Reports | Thoracic Oncology\nRESILIENT Part 2: A Randomized, Open-Label Phase III Study\nof Liposomal Irinotecan Versus Topotecan in Adults With\nRelapsed Small Cell Lung Cancer\nDavidR.Spigel,MD1 ;AfshinDowlati,MD2 ;YuanbinChen,MD3 ;AlejandroNavarro,MD4 ;JamesChih-HsinYang,MD5 ;\nGoranStojanovic,MD6;MariaJove,MD,PhD7 ;PatriciaRich,MD8;ZoranG.Andric,MD9;Yi-LongWu,MD10 ;CharlesM.Rudin,MD,PhD11 ;\nHuanyuChen,PhD12;LiZhang,MPH12;StanleyYeung,PharmD12;FawziBenzaghou,MD12;LuisPaz-Ares,MD13 ;andPaulA.Bunn,MD14 ;\nonbehalfoftheRESILIENTTrialInvestigators\nDOIhttps://doi.org/10.1200/JCO.23.02110', 'PURPOSE ThephaseIIIRESILIENTtrialcomparedsecond-lineliposomalirinotecanwith Appendix\ntopotecaninpatientswithsmallcelllungcancer(SCLC). DataSharing\nPATIENTSAND Patients with SCLC and progre